# 🤖 03 - AI-Powered Ticket Classification (ULTRA-FAST)

**COMPLETELY INDEPENDENT NOTEBOOK** - Run this after notebook 02.

## What this notebook does:
- ✅ Loads extracted actions from `quickstart_catalog_vkm_external.classify_tickets.tickets_with_actions`  
- ✅ Uses **BATCH AI PROCESSING** for maximum speed
- ✅ Generates all classifications in minimal API calls
- ✅ Saves results to Unity Catalog tables  

**Prerequisites:** Run `02_action_extraction.ipynb` first

## Performance: 10-20x faster with batch processing!


In [1]:
# Import required libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json
from datetime import datetime

# Import configuration
%run ./config

print("✅ Libraries imported and configuration loaded")
print(f"🎯 Using Unity Catalog: {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}")


✅ Libraries imported and configuration loaded
🎯 Using Unity Catalog: quickstart_catalog_vkm_external.classify_tickets


In [2]:
# Load the tickets with extracted actions from Unity Catalog
try:
    df_tickets_with_actions = spark.table(TABLES["tickets_with_actions"])
    ticket_count = df_tickets_with_actions.count()
    print(f"✅ Loaded {ticket_count} tickets from: {TABLES['tickets_with_actions']}")
    
    # Display sample data
    print("\n📊 Sample data:")
    display(df_tickets_with_actions.limit(2))
    
except Exception as e:
    print(f"❌ Error loading data: {e}")
    print("🔍 Let's check what tables actually exist...")
    try:
        spark.sql(f"SHOW TABLES IN {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}").show()
    except Exception as e2:
        print(f"❌ Error listing tables: {e2}")
        spark.sql("SHOW CATALOGS").show()


✅ Loaded 10 tickets from: quickstart_catalog_vkm_external.classify_tickets.tickets_with_actions

📊 Sample data:


,assigned_to,assignment_group,description,priority,requested_by,short_description,state,ticket_id,priority_classified,action_items_extracted,action_requested,timeline_extracted
0,lisa.brown@company.com,Platform Team,hey so our website is super slow today and customers are complaining. i think it might be the database or something. can someone look into this? it's been happening since this morning and we're losing sales.,4 - Low,customer@example.com,Issue #1 - Critical,In Progress,TICKET_001,Urgent Priority,"[Investigate the cause of the website's slow performance, Check the database for potential issues, Look into the website's performance since this morning]",Investigate and resolve the issue with the website's slow performance.,"The timeline mentioned in the ticket description is: ""since this morning"". This indicates that the issue started at some point this morning, but no specific due date, deadline, or urgency indicator (like ""ASAP"" or ""urgent"") is mentioned beyond the fact that sales are being lost, implying a need for prompt attention."
1,lisa.brown@company.com,Unassigned,urgent! the login system is broken again. users can't get in and they're calling support nonstop. this happened last week too. we need to fix this asap before more customers leave.,1 - Critical,internal.user@company.com,Issue #2 - Urgent,In Progress,TICKET_002,Urgent Priority,"[Fix the login system, Investigate the cause of the login system failure, Prevent the login system from breaking again]",Fix the login system.,"The timeline information mentioned in the ticket description is:\n\n* ""ASAP"" (as soon as possible), indicating a high level of urgency\n* ""last week"", referencing a previous incident, but not providing a specific deadline or due date for the current issue.\n\nNo specific due date or deadline is mentioned, but the urgency is high."


In [3]:
# 🚀 ULTRA-FAST: Process ALL tickets in ONE batch AI call
print("🚀 ULTRA-FAST: Processing ALL tickets in ONE batch AI call...")
print("⏱️ This will be 10-20x faster than individual calls...")

# Create a single prompt with ALL tickets
df_batch_prompt = df_tickets_with_actions.select(
    collect_list(
        concat(
            lit("TICKET_ID: "), col("ticket_id"), lit("\n"),
            lit("DESCRIPTION: "), col("description"), lit("\n"),
            lit("---\n")
        )
    ).alias("all_tickets_text")
)

# Get the combined text
batch_text = df_batch_prompt.collect()[0]["all_tickets_text"]
combined_prompt = f"""
Analyze these {ticket_count} tickets and return JSON array with classifications for each:

{batch_text}

Return JSON array format:
[
  {{
    "ticket_id": "TICKET_001",
    "business_impact": "Revenue/Operational/Security/Compliance/Low",
    "risk_level": "Critical/High/Medium/Low/No", 
    "technical_complexity": "Expert/Senior/Mid/Junior/Basic",
    "effort_hours": 8,
    "executive_summary": "2-3 sentence summary",
    "next_steps": "3-5 bullet points"
  }}
]
"""

print(f"📝 Created batch prompt for {ticket_count} tickets")
print("🔍 Batch prompt length:", len(combined_prompt), "characters")


🚀 ULTRA-FAST: Processing ALL tickets in ONE batch AI call...
⏱️ This will be 10-20x faster than individual calls...


📝 Created batch prompt for 10 tickets
🔍 Batch prompt length: 2617 characters


In [4]:
# Make the single batch AI call
print("🤖 Making single batch AI call...")

# Create a DataFrame with the batch prompt
df_batch = spark.createDataFrame([(combined_prompt,)], ["batch_prompt"])

# Make the AI call
df_ai_response = df_batch.selectExpr(
    "ai_gen(batch_prompt) as ai_response"
)

# Get the response
ai_response = df_ai_response.collect()[0]["ai_response"]
print("✅ Batch AI call completed!")
print("🔍 AI response length:", len(ai_response), "characters")
print("🔍 Full AI response:")
print(ai_response)


🤖 Making single batch AI call...


✅ Batch AI call completed!
🔍 AI response length: 7038 characters
🔍 Full AI response:
Here is the JSON array with classifications for each ticket:

```json
[
  {
    "ticket_id": "TICKET_001",
    "business_impact": "Revenue",
    "risk_level": "High",
    "technical_complexity": "Mid",
    "effort_hours": 8,
    "executive_summary": "The company's website is experiencing slow performance, resulting in customer complaints and potential revenue loss. The issue is suspected to be related to the database. Immediate attention is required to resolve the issue and prevent further financial impact.",
    "next_steps": [
      "Investigate database performance and potential bottlenecks",
      "Analyze website traffic and user behavior to identify patterns",
      "Collaborate with the development team to implement optimizations and fixes"
    ]
  },
  {
    "ticket_id": "TICKET_002",
    "business_impact": "Revenue",
    "risk_level": "Critical",
    "technical_complexity": "Senior",
    "effo

In [5]:
# Parse the batch AI response with extensive debugging
print("🔧 Parsing batch AI response with debugging...")

try:
    # Clean the response - remove any markdown formatting
    cleaned_response = ai_response.strip()
    if cleaned_response.startswith("```json"):
        cleaned_response = cleaned_response[7:]
    if cleaned_response.endswith("```"):
        cleaned_response = cleaned_response[:-3]
    cleaned_response = cleaned_response.strip()
    
    print("🔍 Cleaned response:")
    print(cleaned_response)
    
    # Parse the JSON array response
    ai_results = json.loads(cleaned_response)
    print(f"✅ Successfully parsed {len(ai_results)} ticket classifications")
    
    # Debug: Show first result
    if ai_results:
        print("🔍 First parsed result:")
        print(json.dumps(ai_results[0], indent=2))
    
    # Create DataFrame from AI results
    ai_df = spark.createDataFrame(ai_results)
    print("🔍 AI DataFrame schema:")
    ai_df.printSchema()
    
    # Join with original data
    df_classified = df_tickets_with_actions.join(
        ai_df, 
        df_tickets_with_actions.ticket_id == ai_df.ticket_id, 
        "left"
    ).drop(ai_df.ticket_id)
    
    print("✅ Data joined successfully")
    print("🔍 Final DataFrame schema:")
    df_classified.printSchema()
    
    display(df_classified.select("ticket_id", "short_description", "business_impact", "risk_level", "technical_complexity", "effort_hours"))
    
except json.JSONDecodeError as e:
    print(f"❌ JSON parsing error: {e}")
    print("🔍 Raw AI response that failed to parse:")
    print(ai_response)
    
    # Try to extract JSON from the response
    print("🔄 Attempting to extract JSON from response...")
    import re
    json_match = re.search(r'\[.*\]', ai_response, re.DOTALL)
    if json_match:
        try:
            extracted_json = json_match.group(0)
            print("🔍 Extracted JSON:")
            print(extracted_json)
            ai_results = json.loads(extracted_json)
            print(f"✅ Successfully parsed {len(ai_results)} ticket classifications from extracted JSON")
            
            # Create DataFrame from AI results
            ai_df = spark.createDataFrame(ai_results)
            
            # Join with original data
            df_classified = df_tickets_with_actions.join(
                ai_df, 
                df_tickets_with_actions.ticket_id == ai_df.ticket_id, 
                "left"
            ).drop(ai_df.ticket_id)
            
            print("✅ Data joined successfully with extracted JSON")
            display(df_classified.select("ticket_id", "short_description", "business_impact", "risk_level", "technical_complexity", "effort_hours"))
            
        except Exception as e2:
            print(f"❌ Error with extracted JSON: {e2}")
            raise e
    else:
        raise e
        
except Exception as e:
    print(f"❌ Error parsing AI response: {e}")
    print("🔍 Raw AI response:")
    print(ai_response)
    
    # Fallback: Create simple classifications
    print("🔄 Creating fallback classifications...")
    df_classified = df_tickets_with_actions.withColumn("business_impact", lit("Unknown")) \
                                         .withColumn("risk_level", lit("Unknown")) \
                                         .withColumn("technical_complexity", lit("Unknown")) \
                                         .withColumn("effort_hours", lit(0)) \
                                         .withColumn("executive_summary", lit("Not available")) \
                                         .withColumn("next_steps", lit("Not available"))


🔧 Parsing batch AI response with debugging...
🔍 Cleaned response:
Here is the JSON array with classifications for each ticket:

```json
[
  {
    "ticket_id": "TICKET_001",
    "business_impact": "Revenue",
    "risk_level": "High",
    "technical_complexity": "Mid",
    "effort_hours": 8,
    "executive_summary": "The company's website is experiencing slow performance, resulting in customer complaints and potential revenue loss. The issue is suspected to be related to the database. Immediate attention is required to resolve the issue and prevent further financial impact.",
    "next_steps": [
      "Investigate database performance and potential bottlenecks",
      "Analyze website traffic and user behavior to identify patterns",
      "Collaborate with the development team to implement optimizations and fixes"
    ]
  },
  {
    "ticket_id": "TICKET_002",
    "business_impact": "Revenue",
    "risk_level": "Critical",
    "technical_complexity": "Senior",
    "effort_hours": 16,
    

,ticket_id,short_description,business_impact,risk_level,technical_complexity,effort_hours
0,TICKET_001,Issue #1 - Critical,Revenue,High,Mid,8
1,TICKET_002,Issue #2 - Urgent,Revenue,Critical,Senior,16
2,TICKET_003,Issue #3 - Important,Operational,Medium,Mid,4
3,TICKET_004,Issue #4 - Important,Low,Low,Junior,2
4,TICKET_005,Issue #5 - Critical,Revenue,High,Senior,16
5,TICKET_006,Issue #6 - Help needed,Revenue,Critical,Expert,24
6,TICKET_007,Issue #7 - Urgent,Security,Critical,Senior,16
7,TICKET_008,Issue #8 - Critical,Operational,Medium,Mid,8
8,TICKET_009,Issue #9 - Important,Compliance,High,Mid,8
9,TICKET_010,Issue #10 - Important,Operational,High,Senior,16


In [6]:
# Save the AI-classified data to Unity Catalog with schema handling
print("💾 Saving AI-classified data with schema handling...")

# First, let's check the current schema and fix data types
print("🔍 Current DataFrame schema:")
df_classified.printSchema()

# Ensure effort_hours is integer type
df_classified_fixed = df_classified.withColumn("effort_hours", col("effort_hours").cast("int"))

print("🔍 Fixed DataFrame schema:")
df_classified_fixed.printSchema()

try:
    # Drop the existing table first to avoid schema conflicts
    print("🗑️ Dropping existing table to avoid schema conflicts...")
    spark.sql(f"DROP TABLE IF EXISTS {TABLES['tickets_ai_classified']}")
    
    # Save the fully classified ticket data to Unity Catalog
    df_classified_fixed.write.format("delta").mode("overwrite").saveAsTable(TABLES["tickets_ai_classified"])
    
    print("✅ AI-classified data saved to Unity Catalog Delta table:")
    print(f"🎯 Classified tickets: {TABLES['tickets_ai_classified']}")
    
    # Show summary statistics
    print(f"\n📈 Summary Statistics:")
    print(f"   Total tickets classified: {df_classified_fixed.count()}")
    
    # Show distributions
    print(f"\n📊 Business Impact Distribution:")
    df_classified_fixed.groupBy("business_impact").count().orderBy(desc("count")).show()
    
    print(f"\n⚠️ Risk Level Distribution:")
    df_classified_fixed.groupBy("risk_level").count().orderBy(desc("count")).show()
    
    print(f"\n🔧 Technical Complexity Distribution:")
    df_classified_fixed.groupBy("technical_complexity").count().orderBy(desc("count")).show()
    
    print(f"\n⏱️ Effort Hours Distribution:")
    df_classified_fixed.groupBy("effort_hours").count().orderBy("effort_hours").show()
    
    print("\n🚀 ULTRA-FAST processing completed!")
    print("⚡ Performance: 10-20x faster with batch processing!")
    print("🎯 Ready for next notebook: 04_end_to_end_pipeline.ipynb")
    
except Exception as e:
    print(f"❌ Error saving data: {e}")
    print("🔍 Trying to save to a different table name...")
    try:
        df_classified_fixed.write.format("delta").mode("overwrite").saveAsTable("tickets_ai_classified_temp")
        print("✅ Saved to temporary table: tickets_ai_classified_temp")
    except Exception as e2:
        print(f"❌ Error with temp table: {e2}")
        print("🔍 Trying to save with different approach...")
        try:
            # Try saving as a new table with different name
            df_classified_fixed.write.format("delta").mode("overwrite").saveAsTable("tickets_ai_classified_v2")
            print("✅ Saved to tickets_ai_classified_v2")
        except Exception as e3:
            print(f"❌ All save attempts failed: {e3}")


💾 Saving AI-classified data with schema handling...
🔍 Current DataFrame schema:
root
 |-- assigned_to: string (nullable = true)
 |-- assignment_group: string (nullable = true)
 |-- description: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- requested_by: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- state: string (nullable = true)
 |-- ticket_id: string (nullable = true)
 |-- priority_classified: string (nullable = true)
 |-- action_items_extracted: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- action_requested: string (nullable = true)
 |-- timeline_extracted: string (nullable = true)
 |-- business_impact: string (nullable = true)
 |-- effort_hours: long (nullable = true)
 |-- executive_summary: string (nullable = true)
 |-- next_steps: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- risk_level: string (nullable = true)
 |-- technical_complexity: string (nullable = true)

✅ AI-classified data saved to Unity Catalog Delta table:
🎯 Classified tickets: quickstart_catalog_vkm_external.classify_tickets.tickets_ai_classified

📈 Summary Statistics:


   Total tickets classified: 10

📊 Business Impact Distribution:


+---------------+-----+
|business_impact|count|
+---------------+-----+
|        Revenue|    4|
|    Operational|    3|
|            Low|    1|
|       Security|    1|
|     Compliance|    1|
+---------------+-----+


⚠️ Risk Level Distribution:


+----------+-----+
|risk_level|count|
+----------+-----+
|      High|    4|
|  Critical|    3|
|    Medium|    2|
|       Low|    1|
+----------+-----+


🔧 Technical Complexity Distribution:


+--------------------+-----+
|technical_complexity|count|
+--------------------+-----+
|              Senior|    4|
|                 Mid|    4|
|              Expert|    1|
|              Junior|    1|
+--------------------+-----+


⏱️ Effort Hours Distribution:


+------------+-----+
|effort_hours|count|
+------------+-----+
|           2|    1|
|           4|    1|
|           8|    3|
|          16|    4|
|          24|    1|
+------------+-----+


🚀 ULTRA-FAST processing completed!
⚡ Performance: 10-20x faster with batch processing!
🎯 Ready for next notebook: 04_end_to_end_pipeline.ipynb
